In [ ]:
# EXPERIMENT: ATTACK VS COMMUNICATION ROUND 
from torch.utils.data import Subset, TensorDataset
from skimage.metrics import structural_similarity as ssim
import pandas as pd
import numpy as np
import os

#Goal: compare reconstruction quality across rounds with multiple seeds and multiple secret samples.

target_rounds = [1, 5, 10, 20, 50]
seeds = [0, 1, 2, 3, 4]
secret_indices = [7, 13, 25]
viz_secret_index = secret_indices[0]

pretrain_subset_size = 1024
pretrain_batch_size = 64
client_lr = 0.1

subset_indices = list(range(min(pretrain_subset_size, len(dataset))))
pretrain_subset = Subset(dataset, subset_indices)

def make_pretrain_loader(seed: int):
    g = torch.Generator()
    g.manual_seed(seed)
    return DataLoader(
        pretrain_subset,
        batch_size=pretrain_batch_size,
        shuffle=True,
        generator=g,
    )

def run_attack_at_round(target_round: int, seed: int, secret_index: int):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    pretrain_loader = make_pretrain_loader(seed)

    
    secret_x_cpu, secret_y_raw = dataset[secret_index]
    secret_x_cpu = secret_x_cpu.unsqueeze(0)
    secret_y_cpu = torch.tensor([secret_y_raw], dtype=torch.long)

    
    base_model_tmp = LeNet().to(device)
    client_tmp = FedAVGClient(copy.deepcopy(base_model_tmp), user_id=0, lr=client_lr, device=device)

    if target_round > 1:
        standard_server_tmp = FedAVGServer([client_tmp], copy.deepcopy(base_model_tmp), device=device)
        api_pretrain_tmp = FedAVGAPI(
            standard_server_tmp,
            [client_tmp],
            criterion,
            [optim.SGD(client_tmp.parameters(), lr=client_lr)],
            [pretrain_loader],
            num_communication=target_round - 1,
            local_epoch=1,
            use_gradients=False,
            device=device,
        )
        api_pretrain_tmp.run()
        attack_init_model = copy.deepcopy(standard_server_tmp.server_model)
    else:
        attack_init_model = copy.deepcopy(base_model_tmp)


    local_dataloaders_tmp = [DataLoader(TensorDataset(secret_x_cpu, secret_y_cpu), batch_size=1, shuffle=False)]
    attacking_server_tmp = AttackingServer([client_tmp], attack_init_model, device=device)
    api_attack_tmp = FedAVGAPI(
        attacking_server_tmp,
        [client_tmp],
        criterion,
        [optim.SGD(client_tmp.parameters(), lr=client_lr)],
        local_dataloaders_tmp,
        num_communication=1,
        local_epoch=1,
        use_gradients=True,
        device=device,
    )
    api_attack_tmp.run()

    recon_x, recon_y = attacking_server_tmp.attack_results[-1][0]

    if recon_y.ndim == 0:
        pred_label = int(recon_y.item())
    elif recon_y.ndim == 1:
        pred_label = int(torch.argmax(recon_y).item())
    else:
        pred_label = int(torch.argmax(recon_y[0]).item())

    recon_img = recon_x[0][0].detach().cpu().numpy()
    secret_img = secret_x_cpu[0][0].detach().cpu().numpy()

    mse = float(np.mean((recon_img - secret_img) ** 2))
    ssim_score = float(ssim(secret_img, recon_img, data_range=1.0))
    label_ok = int(pred_label == int(secret_y_cpu.item()))

    return {
        "round": target_round,
        "seed": seed,
        "secret_index": secret_index,
        "pred_label": pred_label,
        "true_label": int(secret_y_cpu.item()),
        "label_ok": label_ok,
        "mse": mse,
        "ssim": ssim_score,
        "recon_x": recon_x,
        "secret_x_cpu": secret_x_cpu,
    }


raw_results = []
for idx in secret_indices:
    for r in target_rounds:
        for s in seeds:
            print(f"Running attack at idx {idx}, round {r}, seed {s} ...")
            raw_results.append(run_attack_at_round(r, s, idx))

#TABLES
raw_rows = [
    {
        "secret_index": x["secret_index"],
        "round": x["round"],
        "seed": x["seed"],
        "true_label": x["true_label"],
        "pred_label": x["pred_label"],
        "label_ok": x["label_ok"],
        "mse": x["mse"],
        "ssim": x["ssim"],
    }
    for x in raw_results
]
df_raw = pd.DataFrame(raw_rows)

# Mean/std summaries
df_summary = df_raw.groupby("round", as_index=False).agg(
    mse_mean=("mse", "mean"),
    mse_std=("mse", "std"),
    ssim_mean=("ssim", "mean"),
    ssim_std=("ssim", "std"),
    label_acc=("label_ok", "mean"),
)
df_summary["label_acc"] = 100.0 * df_summary["label_acc"]

df_summary_by_secret = df_raw.groupby(["secret_index", "round"], as_index=False).agg(
    mse_mean=("mse", "mean"),
    mse_std=("mse", "std"),
    ssim_mean=("ssim", "mean"),
    ssim_std=("ssim", "std"),
    label_acc=("label_ok", "mean"),
)
df_summary_by_secret["label_acc"] = 100.0 * df_summary_by_secret["label_acc"]

# Robust summaries (median + quartiles + IQR)
robust_rows = []
for r, grp in df_raw.groupby("round"):
    mse_q25 = float(grp["mse"].quantile(0.25))
    mse_q75 = float(grp["mse"].quantile(0.75))
    ssim_q25 = float(grp["ssim"].quantile(0.25))
    ssim_q75 = float(grp["ssim"].quantile(0.75))
    robust_rows.append({
        "round": int(r),
        "mse_median": float(grp["mse"].median()),
        "mse_q25": mse_q25,
        "mse_q75": mse_q75,
        "mse_iqr": mse_q75 - mse_q25,
        "ssim_median": float(grp["ssim"].median()),
        "ssim_q25": ssim_q25,
        "ssim_q75": ssim_q75,
        "ssim_iqr": ssim_q75 - ssim_q25,
        "label_acc": 100.0 * float(grp["label_ok"].mean()),
    })
df_summary_robust = pd.DataFrame(robust_rows).sort_values("round").reset_index(drop=True)

robust_by_secret_rows = []
for (idx, r), grp in df_raw.groupby(["secret_index", "round"]):
    mse_q25 = float(grp["mse"].quantile(0.25))
    mse_q75 = float(grp["mse"].quantile(0.75))
    ssim_q25 = float(grp["ssim"].quantile(0.25))
    ssim_q75 = float(grp["ssim"].quantile(0.75))
    robust_by_secret_rows.append({
        "secret_index": int(idx),
        "round": int(r),
        "mse_median": float(grp["mse"].median()),
        "mse_q25": mse_q25,
        "mse_q75": mse_q75,
        "mse_iqr": mse_q75 - mse_q25,
        "ssim_median": float(grp["ssim"].median()),
        "ssim_q25": ssim_q25,
        "ssim_q75": ssim_q75,
        "ssim_iqr": ssim_q75 - ssim_q25,
        "label_acc": 100.0 * float(grp["label_ok"].mean()),
    })
df_summary_robust_by_secret = pd.DataFrame(robust_by_secret_rows).sort_values(["secret_index", "round"]).reset_index(drop=True)

#SAVE CSV OUTPUTS
os.makedirs("results/baseline", exist_ok=True)
raw_csv_path = "results/baseline/attack_vs_round_multisecret_raw.csv"
summary_csv_path = "results/baseline/attack_vs_round_multisecret_summary.csv"
summary_by_secret_csv_path = "results/baseline/attack_vs_round_multisecret_summary_by_secret.csv"
summary_robust_csv_path = "results/baseline/attack_vs_round_multisecret_summary_robust.csv"
summary_robust_by_secret_csv_path = "results/baseline/attack_vs_round_multisecret_summary_robust_by_secret.csv"
df_raw.to_csv(raw_csv_path, index=False)
df_summary.to_csv(summary_csv_path, index=False)
df_summary_by_secret.to_csv(summary_by_secret_csv_path, index=False)
df_summary_robust.to_csv(summary_robust_csv_path, index=False)
df_summary_robust_by_secret.to_csv(summary_robust_by_secret_csv_path, index=False)


viz_candidates = [x for x in raw_results if x["secret_index"] == viz_secret_index]
if len(viz_candidates) > 0:
    best_by_round = {}
    for r in target_rounds:
        candidates = [x for x in viz_candidates if x["round"] == r]
        best_by_round[r] = min(candidates, key=lambda x: x["mse"])

    secret_ref = viz_candidates[0]["secret_x_cpu"]
    true_ref_label = int(viz_candidates[0]["true_label"])

    n_cols = len(target_rounds) + 1
    plt.figure(figsize=(3 * n_cols, 3.8))

    plt.subplot(1, n_cols, 1)
    plt.title(f"Original idx {viz_secret_index}\nLabel: {true_ref_label}")
    plt.imshow(secret_ref[0][0].detach().numpy(), cmap="gray")
    plt.axis("off")

    for i, r in enumerate(target_rounds, start=2):
        best = best_by_round[r]
        srow = df_summary_by_secret[
            (df_summary_by_secret["secret_index"] == viz_secret_index) &
            (df_summary_by_secret["round"] == r)
        ].iloc[0]
        plt.subplot(1, n_cols, i)
        plt.title(
            f"Round {r}\n"
            f"Best Pred: {best['pred_label']}\n"
            f"MSE: {srow['mse_mean']:.4g}+-{srow['mse_std']:.2g}\n"
            f"SSIM: {srow['ssim_mean']:.3f}+-{srow['ssim_std']:.2f}"
        )
        plt.imshow(best["recon_x"][0][0].detach().cpu().numpy(), cmap="gray")
        plt.axis("off")

    plt.tight_layout()
    plt.show()


print("\nGlobal summary by round (all secrets + all seeds):")
for _, row in df_summary.sort_values("round").iterrows():
    print(
        f"Round {int(row['round']):>3} | "
        f"MSE {row['mse_mean']:.6f} +- {row['mse_std']:.6f} | "
        f"SSIM {row['ssim_mean']:.4f} +- {row['ssim_std']:.4f} | "
        f"Label Acc {row['label_acc']:.1f}%"
    )

print("\nRobust summary by round (median [p25, p75]):")
for _, row in df_summary_robust.sort_values("round").iterrows():
    print(
        f"Round {int(row['round']):>3} | "
        f"MSE {row['mse_median']:.6f} [{row['mse_q25']:.6f}, {row['mse_q75']:.6f}] | "
        f"SSIM {row['ssim_median']:.4f} [{row['ssim_q25']:.4f}, {row['ssim_q75']:.4f}] | "
        f"Label Acc {row['label_acc']:.1f}%"
    )

print("\nSaved raw results: " + raw_csv_path)
print("Saved global summary: " + summary_csv_path)
print("Saved by-secret summary: " + summary_by_secret_csv_path)
print("Saved robust summary: " + summary_robust_csv_path)
print("Saved robust by-secret summary: " + summary_robust_by_secret_csv_path)